# Installations

In [1]:
%pip install -q mlflow databricks-sdk

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 113.4 MB/s eta 0:00:0000:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 116.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 79.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 969.1/969.1 kB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 95.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
%pip install cuml-cu12 --extra-index-url=https://pypi.nvidia.com
%pip install optuna

Looking in indexes: https://pypi.org/simple, https://pypi.nvidia.com


In [18]:
import cuml
print("cuML Version:", cuml.__version__)

cuML Version: 26.02.000


# Setup

In [18]:
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# Preprocessing
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# Model Selection
from sklearn.model_selection import StratifiedKFold 

# Models CPU
from sklearn.ensemble import RandomForestClassifier

#GPU
# from cuml.ensemble import RandomForestClassifier
# import cupy as cp

# Tuning
# import optuna
# from optuna.integration import LightGBMPruningCallback

import mlflow
import os
import json

# Metrics 
from sklearn.metrics import balanced_accuracy_score

import warnings
warnings.filterwarnings('ignore')

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
dataPath = '/content/drive/MyDrive/KaggleData/s6e7/'
train = pd.read_csv(dataPath + 'train.csv', low_memory=False)
test = pd.read_csv(dataPath + 'test.csv', low_memory=False)
sample_submission = pd.read_csv(dataPath + 'sample_submission.csv', index_col='id', low_memory=False)

In [6]:
train.head()

,id,health_condition,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
0,0,unhealthy,5.22,70.6,25.66,2174.0,1326.0,19.8,1.86,veg,high,average,sedentary,yes,female
1,1,at-risk,5.53,71.3,25.84,1966.0,9891.0,49.9,1.26,non-veg,low,average,moderate,yes,other
2,2,unhealthy,5.29,75.4,24.54,2688.0,14216.0,38.1,1.60,veg,high,poor,active,yes,male
3,3,unhealthy,4.70,77.2,23.13,2630.0,7174.0,59.9,2.02,veg,high,average,active,occasional,female
4,4,at-risk,7.23,73.4,28.44,2560.0,6584.0,46.0,2.25,veg,NaN,average,sedentary,NaN,male


In [7]:
test.head()

,id,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
0,690088,5.35,64.9,23.48,2745.0,14167.0,59.5,1.86,veg,high,poor,active,occasional,male
1,690089,NaN,83.1,22.42,1773.0,6801.0,24.5,2.40,balanced,high,poor,sedentary,yes,other
2,690090,6.68,59.7,24.14,3040.0,13250.0,48.5,2.76,balanced,medium,poor,active,no,NaN
3,690091,7.13,78.5,26.26,2494.0,6331.0,56.9,2.34,veg,low,good,moderate,yes,other
4,690092,5.49,77.7,23.29,1828.0,13894.0,39.4,2.45,veg,high,average,active,occasional,other


In [9]:
sample_submission.head()

,health_condition
id,
690088,at-risk
690089,at-risk
690090,at-risk
690091,at-risk
690092,at-risk


In [10]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 690088 entries, 0 to 690087
Data columns (total 15 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       690088 non-null  int64  
 1   health_condition         690088 non-null  object 
 2   sleep_duration           614089 non-null  float64
 3   heart_rate               682255 non-null  float64
 4   bmi                      676190 non-null  float64
 5   calorie_expenditure      637235 non-null  float64
 6   step_count               676172 non-null  float64
 7   exercise_duration        683187 non-null  float64
 8   water_intake             646611 non-null  float64
 9   diet_type                683187 non-null  object 
 10  stress_level             607277 non-null  object 
 11  sleep_quality            631757 non-null  object 
 12  physical_activity_level  653467 non-null  object 
 13  smoking_alcohol          661506 non-null  object 
 14  gend

In [11]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 295753 entries, 0 to 295752
Data columns (total 14 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       295753 non-null  int64  
 1   sleep_duration           263182 non-null  float64
 2   heart_rate               292396 non-null  float64
 3   bmi                      289797 non-null  float64
 4   calorie_expenditure      273101 non-null  float64
 5   step_count               289789 non-null  float64
 6   exercise_duration        292795 non-null  float64
 7   water_intake             277120 non-null  float64
 8   diet_type                292795 non-null  object 
 9   stress_level             260263 non-null  object 
 10  sleep_quality            270754 non-null  object 
 11  physical_activity_level  280058 non-null  object 
 12  smoking_alcohol          283504 non-null  object 
 13  gender                   286593 non-null  object 
dtypes: f

# Features

In [6]:
conts = ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure','step_count', 'exercise_duration', 'water_intake']
cats = ['diet_type', 'stress_level', 'sleep_quality', 'physical_activity_level','smoking_alcohol', 'gender']

target = 'health_condition'

# Preprocessing

## Encoding

In [7]:
catOrders = {
  "stress_level": ['low', 'medium', 'high'],
  "sleep_quality": ['poor', 'average', 'good'],
  "physical_activity": ['sedentary','moderate', 'active'],
  "smoking_alcohol": ['no', 'occasional', 'yes']
}

In [8]:
class PreprocessorFactory:
    def __init__(self, conts, nominal_cats, ordinal_mappings):

        self.conts = conts
        self.nominal_cats = nominal_cats
        self.ordinal_mappings = ordinal_mappings

    def build(self, selected_features, 
              cont_imputer=None, 
              cat_imputer=None):
        
        # Set default imputers if none are passed
        if cont_imputer is None:
            cont_imputer = SimpleImputer(strategy='median')
        if cat_imputer is None:
            cat_imputer = SimpleImputer(strategy='constant', fill_value='Missing')
        
        # 1. Filter the feature lists based on what is in selected_features
        active_conts = [col for col in self.conts if col in selected_features]
        active_nominal = [col for col in self.nominal_cats if col in selected_features]
        
        # 2. Filter the ordinal columns AND their mappings together
        active_ordinal = [col for col in self.ordinal_mappings.keys() if col in selected_features]
        active_categories = [self.ordinal_mappings[col] for col in active_ordinal]

        # 3. Build the mini-pipelines dynamically
        transformers = []

        # Only add the continuous pipeline if we actually selected continuous features
        if active_conts:
            num_pipe = Pipeline([('imputer', cont_imputer)])
            transformers.append(('continuous', num_pipe, active_conts))

        # Only add the nominal pipeline if we selected nominal features
        if active_nominal:
            nom_pipe = Pipeline([
                ('imputer', cat_imputer),
                ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
            ])
            transformers.append(('nominal', nom_pipe, active_nominal))

        # Only add the ordinal pipeline if we selected ordinal features
        if active_ordinal:
            ord_pipe = Pipeline([
                ('imputer', cat_imputer),
                ('encoder', OrdinalEncoder(categories=active_categories, handle_unknown='use_encoded_value', unknown_value=-1))
            ])
            transformers.append(('ordinal', ord_pipe, active_ordinal))

        # 4. Assemble and return the final ColumnTransformer
        preprocessor = ColumnTransformer(transformers=transformers, remainder='drop')
        
        # Keep pandas dataframe formatting
        preprocessor.set_output(transform="pandas")
        
        return preprocessor

In [9]:
le  = LabelEncoder()
train[target] = le.fit_transform(train[target])

In [10]:
feature_cols = conts + cats

### testing preprocessor

In [12]:
# Create the factory once
factory = PreprocessorFactory(conts, cats, catOrders)

# Scenario A: Run your baseline with the filtered features
selected_features = ['sleep_duration', 'stress_level', 'physical_activity_level', 'bmi']

dynamic_preprocessor = factory.build(selected_features)

# rf_model = Learner(train, test, target, RandomForestClassifier, selected_features, 'RF_Subset', balanced_accuracy_score, preprocessor=dynamic_preprocessor)
# rf_model.fit()

# # ---------------------------------------------------------

# # Scenario B: Run the same features, but let's test KNN Imputation for continuous!
# from sklearn.impute import KNNImputer

# knn_preprocessor = factory.build(selected_features, cont_imputer=KNNImputer(n_neighbors=5))

# rf_knn_model = Learner(train, test, target, RandomForestClassifier, selected_features, 'RF_KNN_Impute', balanced_accuracy_score, preprocessor=knn_preprocessor)
# rf_knn_model.fit()

In [13]:
t = dynamic_preprocessor.fit_transform(train[feature_cols])
print(t.head())

   continuous__sleep_duration  continuous__bmi  nominal__stress_level  \
0                        5.22            25.66                    1.0   
1                        5.53            25.84                    2.0   
2                        5.29            24.54                    1.0   
3                        4.70            23.13                    1.0   
4                        7.23            28.44                    0.0   

   nominal__physical_activity_level  ordinal__stress_level  
0                               3.0                    2.0  
1                               2.0                    0.0  
2                               1.0                    2.0  
3                               1.0                    2.0  
4                               3.0                   -1.0  


In [42]:
preprocessor.set_output(transform='pandas')

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('custom_ordinal', ...), ('nominal_ordinal', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``tran

In [ ]:
t = preprocessor.fit_transform(train[feature_cols])
print(t.head())

   custom_ordinal__stress_level  custom_ordinal__sleep_quality  \
0                           2.0                            1.0   
1                           0.0                            1.0   
2                           2.0                            0.0   
3                           2.0                            1.0   
4                          -1.0                            1.0   

   custom_ordinal__physical_activity_level  custom_ordinal__smoking_alcohol  \
0                                      0.0                              2.0   
1                                      1.0                              2.0   
2                                      2.0                              2.0   
3                                      2.0                              1.0   
4                                      0.0                             -1.0   

   nominal_ordinal__diet_type  nominal_ordinal__gender  \
0                         3.0                      1.0   
1           

# Modelling

## setup MLflow

In [11]:
with open('/content/drive/MyDrive/KaggleData/mlflow/databricks.json', 'r') as f:
  config = json.load(f)

os.environ["DATABRICKS_HOST"] = config.get("DATABRICKS_HOST", "")
os.environ["DATABRICKS_TOKEN"] = config.get("DATABRICKS_TOKEN", "")
os.environ["MLFLOW_TRACKING_URI"] = "databricks"

In [12]:
user_email = "sameeranc11@gmail.com"
experiment_path = f"/Users/{user_email}/Kaggle_s6e7_Comp"
# mlflow.sklearn.autolog(log_models=False) #removing this because  it's creating conflicts with manual logging.
mlflow.set_experiment(experiment_path)

If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


<Experiment: artifact_location='dbfs:/databricks/mlflow-tracking/1584872737403196', creation_time=1784457247063, effective_trace_archival_retention=None, experiment_id='1584872737403196', last_update_time=1784882087504, lifecycle_stage='active', name='/Users/sameeranc11@gmail.com/Kaggle_s6e7_Comp', tags={'mlflow.experiment.sourceName': '/Users/sameeranc11@gmail.com/Kaggle_s6e7_Comp',
 'mlflow.experimentKind': 'custom_model_development',
 'mlflow.experimentType': 'MLFLOW_EXPERIMENT',
 'mlflow.ownerEmail': 'sameeranc11@gmail.com',
 'mlflow.ownerId': '75150582631674'}, trace_location=None, workspace='default'>

## setup kaggle

In [13]:
with open('/content/drive/MyDrive/KaggleData/kaggle.json', 'r') as f:
  config = json.load(f)

os.environ["KAGGLE_API_TOKEN"] = config.get("KAGGLE_API_TOKEN", "")

In [27]:
class Learner:
    
    def __init__(self, df_train, df_test, target, model, selected_features, note, metric, preprocessor=None):
        self.X_train = df_train[selected_features]
        self.Y_train = df_train[target]
        self.X_test = df_test[selected_features]
        self.note = note
        self.metric = metric
        self.random_state = 42
        self.model = model

        # if preprocessor:
            # preprocessor.set_output(transform='pandas')
        self.preprocessor = preprocessor

    def _log_oob_score(self, estimator, fold_idx):
        """Safely logs OOB score if the model supports it."""
        if hasattr(estimator, "oob_score_"):
            oob = estimator.oob_score_
            mlflow.log_metric("fold_oob_score", oob, step=fold_idx)
            print(f"Fold {fold_idx} OOB Score: {oob}")

    def _plot_feature_importances(self, pipeline, fold_idx):
        """Generates and logs feature importance plot as an artifact."""
        estimator = pipeline.named_steps['pred_model']
        
        if hasattr(estimator, "feature_importances_"):
            # Get the exact column names generated by the preprocessor
            if self.preprocessor:
                cols = self.preprocessor.get_feature_names_out()
            else:
                cols = self.X_train.columns

            fi = pd.DataFrame({'cols': cols, 'imp': estimator.feature_importances_})
            fi = fi.sort_values('imp', ascending=False).head(30) # Top 30

            plt.figure(figsize=(10, 6))
            plt.barh(fi['cols'], fi['imp'], color='teal')
            plt.gca().invert_yaxis()
            plt.title(f"Feature Importance - Fold {fold_idx}")
            
            # Save and log to MLflow
            plot_path = f"feature_importance_fold_{fold_idx}.png"
            plt.savefig(plot_path, bbox_inches='tight')
            plt.close()
            mlflow.log_artifact(plot_path)

    def _plot_score_vs_trees(self, pipeline, x_valid, y_valid, fold_idx):
        """Calculates cumulative score as trees increase for Random Forest."""
        estimator = pipeline.named_steps['pred_model']
        
        if hasattr(estimator, "estimators_"):
            # Transform the validation data first so the raw trees can read it
            if self.preprocessor:
                x_valid_transformed = self.preprocessor.transform(x_valid)
            else:
                x_valid_transformed = x_valid

            # Get predictions from every single tree
            tree_preds = [tree.predict(x_valid_transformed) for tree in estimator.estimators_]
            tree_preds = np.array(tree_preds)
            
            # Calculate cumulative mean prediction and score it
            scores = []
            for i in range(1, len(tree_preds) + 1):
                cumulative_pred = np.round(tree_preds[:i].mean(axis=0)) # Majority vote
                score = self.metric(y_valid, cumulative_pred)
                scores.append(score)

            plt.figure(figsize=(8, 4))
            plt.plot(range(1, len(scores) + 1), scores, color='blue')
            plt.title(f"Score vs Number of Trees - Fold {fold_idx}")
            plt.xlabel("Number of Trees")
            plt.ylabel("Validation Score")
            
            plot_path = f"score_vs_trees_fold_{fold_idx}.png"
            plt.savefig(plot_path, bbox_inches='tight')
            plt.close()
            mlflow.log_artifact(plot_path)
    
    def fit(self, trainingMode='standard', params=None, ploting={"fi":False, "score_vs_trees":False}):
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=self.random_state)
        fold_accuracy = []
        train_accuracy = []
        test_preds_list = []
        
        run_name = f"{self.note}_{trainingMode}"
        with mlflow.start_run(run_name=run_name, nested=(trainingMode == 'tuning')):
            
            mlflow.log_param("model", self.model.__name__)
            if self.preprocessor:
                mlflow.log_param("preprocessor", self.preprocessor.__class__.__name__)
            if params:
                mlflow.log_params(params)
                
            for i, (train_index, test_index) in enumerate(skf.split(self.X_train, self.Y_train)):
                print(f"Training fold {i+1}...")
                x_train_fold, x_test_fold = self.X_train.iloc[train_index], self.X_train.iloc[test_index]
                y_train_fold, y_test_fold = self.Y_train.iloc[train_index], self.Y_train.iloc[test_index]
            
                model_instance = self.model(**params) if params else self.model(random_state=self.random_state)
                
                if self.preprocessor:
                    pipeline = Pipeline([
                        ('preprocessor', self.preprocessor),
                        ('pred_model', model_instance)
                    ])
                else:
                    pipeline = model_instance
                    
                pipeline.fit(x_train_fold, y_train_fold)
                
                model_oof_preds = pipeline.predict(x_test_fold)
                OOFaccScore = self.metric(y_test_fold, model_oof_preds)
                fold_accuracy.append(OOFaccScore)

                train_preds = pipeline.predict(x_train_fold)
                train_score = self.metric(y_train_fold, train_preds)
                train_accuracy.append(train_score)
                
                mlflow.log_metric(f"val_acc", OOFaccScore, step=i+1)
                mlflow.log_metric(f"train_acc", train_score, step=i+1)
                print(f"Fold {i+1} ==> OOF score: {OOFaccScore}")

                if trainingMode != 'tuning':
                    raw_estimator = pipeline.named_steps['pred_model'] if self.preprocessor else pipeline
                    self._log_oob_score(raw_estimator, i+1)
                    self._plot_feature_importances(pipeline, i+1) if ploting.get("fi", False) else None
                    self._plot_score_vs_trees(pipeline, x_test_fold, y_test_fold, i+1) if ploting.get("score_vs_trees", False) else None
                    fold_test_probs = pipeline.predict_proba(self.X_test)
                    test_preds_list.append(fold_test_probs)
        
            avgAcc = np.mean(fold_accuracy)
            stdAcc = np.std(fold_accuracy)
            avgTrainAcc = np.mean(train_accuracy)
            stdTrainAcc = np.std(train_accuracy)

            mlflow.log_metric("cv_mean_val_acc", avgAcc)
            mlflow.log_metric("cv_std_val_acc", stdAcc)
            mlflow.log_metric("cv_mean_train_acc", avgTrainAcc)
            mlflow.log_metric("cv_std_train_acc", stdTrainAcc)
            
            print(f"CV Mean Accuracy: {avgAcc:.4f} | Std: {stdAcc:.4f}")
            print('---------------------------------------------------------------')
            
            if trainingMode != 'tuning':
                mean_test_probs = np.mean(test_preds_list, axis=0)
                self.final_test_predictions = np.argmax(mean_test_probs, axis=1)

            return avgAcc

    def tune(self, study_name, n_trials, get_params_func, db_path):
        
        self.tuning_study = optuna.create_study(
            direction='maximize',
            study_name=study_name,
            storage=f'sqlite:///{db_path}', 
            load_if_exists=True
        )
        
        def objective(trial):
            params = get_params_func(trial)
            
            avg_val_acc = self.fit(trainingMode='tuning', params=params)
            
            return avg_val_acc

        run_name = f"{self.note}_Optuna_Search"
        with mlflow.start_run(run_name=run_name):
            
            self.tuning_study.optimize(
                objective, 
                n_trials=n_trials,
                show_progress_bar=True
            )
        
        print(f"Best Params: {self.tuning_study.best_params}")
        return self.tuning_study.best_params

    def fast_gpu_tune(self, study_name, n_trials, get_params_func, db_path):
        
        if self.preprocessor:
            X_processed_cpu = self.preprocessor.fit_transform(self.X_train)
        else:
            X_processed_cpu = self.X_train.copy()
            
        X_processed_cpu = X_processed_cpu.astype('float32')
        X_gpu = cp.array(X_processed_cpu)
        
        y_cpu = self.Y_train.to_numpy()
        y_gpu = cp.array(y_cpu.astype('int32'))

        self.tuning_study = optuna.create_study(
            direction='maximize',
            study_name=study_name,
            storage=f'sqlite:///{db_path}', 
            load_if_exists=True
        )
        
        def objective(trial):
            params = get_params_func(trial)
            
            with mlflow.start_run(run_name=f"Trial_{trial.number}", nested=True):
                
                mlflow.log_param("trial_number", trial.number)
                mlflow.log_params(params)
                
                skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=self.random_state)
                fold_accuracy = []
                train_accuracy = []

                for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X_processed_cpu, y_cpu)):
                    X_train_fold, X_val_fold = X_gpu[train_idx], X_gpu[val_idx]
                    y_train_fold, y_val_fold = y_gpu[train_idx], y_gpu[val_idx]
                
                    model_instance = self.model(**params)
                    model_instance.fit(X_train_fold, y_train_fold)
                    
                    val_preds_gpu = model_instance.predict(X_val_fold)
                    val_preds_cpu = val_preds_gpu.get() 

                    test_preds_gpu = model_instance.predict(X_train_fold)  
                    train_preds_cpu = test_preds_gpu.get()
                    
                    OOF_val_score = self.metric(y_cpu[val_idx], val_preds_cpu)
                    fold_accuracy.append(OOF_val_score)

                    train_score = self.metric(y_cpu[train_idx], train_preds_cpu)
                    train_accuracy.append(train_score)
                    
                    mlflow.log_metric("val_acc", OOF_val_score, step=fold_idx + 1)
                    mlflow.log_metric("train_acc", train_score, step=fold_idx + 1)

                mean_val_acc = np.mean(fold_accuracy)
                mean_train_acc = np.mean(train_accuracy)

                mlflow.log_metric("cv_mean_val_acc", mean_val_acc)
                mlflow.log_metric("cv_std_val_acc", np.std(fold_accuracy))
                mlflow.log_metric("cv_mean_train_acc", mean_train_acc)
                mlflow.log_metric("cv_std_train_acc", np.std(train_accuracy))
            return mean_val_acc

        run_name = f"{self.note}_Fast_GPU_Tune"
        with mlflow.start_run(run_name=run_name):
            
            self.tuning_study.optimize(
                objective, 
                n_trials=n_trials,
                show_progress_bar=True
            )
            
            mlflow.log_params(self.tuning_study.best_params)
            mlflow.log_metric("best_cv_mean_val_acc", self.tuning_study.best_value)
        
        print(f"Best Params: {self.tuning_study.best_params}")
        return self.tuning_study.best_params

In [15]:
def submit_preds(model, target, fileName):   
  final_string_predictions = le.inverse_transform(model.final_test_predictions)
  sample_submission[target] = final_string_predictions
  sample_submission.to_csv(dataPath + fileName)

## Basic RF

In [28]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

basic_preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, conts),
        ("cat", categorical_transformer, cats)
    ]
)

In [21]:
basic_preprocessor.set_output(transform="pandas")

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median'))]),
                                 ['sleep_duration', 'heart_rate', 'bmi',
                                  'calorie_expenditure', 'step_count',
                                  'exercise_duration', 'water_intake']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(fill_value='Missing',
                                                                strategy='constant')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['diet_type', 'stress_level', 'sleep_quality',
                                  'physical_activity_level', 'smoking_alcohol',
                                  'gender'])])

In [ ]:
%%time
# factory = PreprocessorFactory(conts, cats, catOrders)
# dynamic_preprocessor = factory.build(feature_cols)

rf_model = Learner(train, test, target, RandomForestClassifier, feature_cols, 'RF_Baseline_ClassBalanced_Ordinal', balanced_accuracy_score, preprocessor=dynamic_preprocessor)
rf_model.fit(trainingMode='standard', params={'class_weight': 'balanced', 'n_jobs': -1}, ploting={"fi":True, "score_vs_trees":True})

Training fold 1...
Fold 1 ==> OOF score: 0.8645287941391326
Training fold 2...
Fold 2 ==> OOF score: 0.8694700333056669
Training fold 3...
Fold 3 ==> OOF score: 0.8683446480390747
Training fold 4...
Fold 4 ==> OOF score: 0.8637652985922354
Training fold 5...
Fold 5 ==> OOF score: 0.8655419253729525
CV Mean Accuracy: 0.8663 | Std: 0.0022
---------------------------------------------------------------
🏃 View run RF_Baseline_ClassBalanced_Ordinal_standard at: https://dbc-36754e4d-ad0b.cloud.databricks.com/ml/experiments/1584872737403196/runs/874cedeef9184875bb56fac64a207512
🧪 View experiment at: https://dbc-36754e4d-ad0b.cloud.databricks.com/ml/experiments/1584872737403196
CPU times: user 19min 30s, sys: 3.02 s, total: 19min 33s
Wall time: 11min 17s


np.float64(0.8663301398898124)

In [29]:
rf_model = Learner(train, test, target, RandomForestClassifier, feature_cols, 'RF_Baseline_ClassBalanced_OneHot', balanced_accuracy_score, preprocessor=basic_preprocessor)
rf_model.fit(trainingMode='standard', params={'class_weight': 'balanced', 'n_jobs': -1}, ploting={"fi":True, "score_vs_trees":True})

Training fold 1...
Fold 1 ==> OOF score: 0.8627837512231104
Training fold 2...
Fold 2 ==> OOF score: 0.8670132338309856
Training fold 3...
Fold 3 ==> OOF score: 0.8661038928220633
Training fold 4...
Fold 4 ==> OOF score: 0.8613287524034202
Training fold 5...
Fold 5 ==> OOF score: 0.863438575405281
CV Mean Accuracy: 0.8641 | Std: 0.0021
---------------------------------------------------------------
🏃 View run RF_Baseline_ClassBalanced_OneHot_standard at: https://dbc-36754e4d-ad0b.cloud.databricks.com/ml/experiments/1584872737403196/runs/40b7231ba3ca42089ceb3fb659ee2f28
🧪 View experiment at: https://dbc-36754e4d-ad0b.cloud.databricks.com/ml/experiments/1584872737403196


np.float64(0.8641336411369721)

In [10]:
%%time
rf_model = Learner(train, test, target, RandomForestClassifier, feature_cols, 'RF_Baseline_cpu_parallel', balanced_accuracy_score, preprocessor=preprocessor)
rf_model.fit(trainingMode='standard', params={'n_jobs': -1})

Training fold 1...
Fold 1 ==> OOF score: 0.8640426126450014
Training fold 2...
Fold 2 ==> OOF score: 0.867509427211929
Training fold 3...
Fold 3 ==> OOF score: 0.8671821906859297
Training fold 4...
Fold 4 ==> OOF score: 0.8628230962474835
Training fold 5...
Fold 5 ==> OOF score: 0.8651312766108487
CV Mean Accuracy: 0.8653 | Std: 0.0018
---------------------------------------------------------------
🏃 View run RF_Baseline_cpu_parallel_standard at: https://dbc-36754e4d-ad0b.cloud.databricks.com/ml/experiments/1584872737403196/runs/9c8d8cb6bff84c3ca4738b31f509e272
🧪 View experiment at: https://dbc-36754e4d-ad0b.cloud.databricks.com/ml/experiments/1584872737403196
CPU times: user 25min 29s, sys: 5.42 s, total: 25min 34s
Wall time: 15min 51s


np.float64(0.8653377206802384)

In [ ]:
submit_preds(rf_model, target, 'rf_baseline_submission.csv')

## top features only

In [ ]:
%%time

factory = PreprocessorFactory(conts, cats, catOrders)
selected_features = ['sleep_duration', 'stress_level', 'physical_activity_level', 'bmi']
dynamic_preprocessor = factory.build(selected_features)

rf_model = Learner(train, test, target, RandomForestClassifier, selected_features, 'RF_Subset_top_features_4', balanced_accuracy_score, preprocessor=dynamic_preprocessor)
rf_model.fit(trainingMode='standard', params={'n_jobs': -1, 'n_estimators': 85,})

Training fold 1...
Fold 1 ==> OOF score: 0.8705955723809224
Training fold 2...
Fold 2 ==> OOF score: 0.8727096381998002
Training fold 3...
Fold 3 ==> OOF score: 0.8720419834058676
Training fold 4...
Fold 4 ==> OOF score: 0.8699351949796834
Training fold 5...
Fold 5 ==> OOF score: 0.8685471756816314
CV Mean Accuracy: 0.8708 | Std: 0.0015
---------------------------------------------------------------
🏃 View run RF_Subset_top_features_4_standard at: https://dbc-36754e4d-ad0b.cloud.databricks.com/ml/experiments/1584872737403196/runs/a7089789949141439f9c5cfbd2587634
🧪 View experiment at: https://dbc-36754e4d-ad0b.cloud.databricks.com/ml/experiments/1584872737403196
CPU times: user 13min 17s, sys: 2.24 s, total: 13min 19s
Wall time: 8min 22s


np.float64(0.8707659129295809)

In [ ]:
submit_preds(rf_model, target, 'rf_top_features_4_submission.csv')+

In [26]:
print(dataPath + 'rf_top_features_submission.csv')

/content/drive/MyDrive/KaggleData/s6e7/rf_top_features_submission.csv


In [39]:
!kaggle competitions submit -c playground-series-s6e7 -f /content/drive/MyDrive/KaggleData/s6e7/RF_Subset_top_features_4_tuned_bp1.csv  -m "RF_Subset_top_features_4_tuned_bp1"

100% 4.21M/4.21M [00:02<00:00, 1.64MB/s]
Successfully submitted to Predicting Student Health Risk

## Fine Tuning RF

In [25]:
def rf_param_space(trial):
    """Defines the hyperparameter space for RandomForestClassifier."""
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 200, 50),
        'max_depth': trial.suggest_int('max_depth', 15, 40, 5),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 2, 50, 4),
        'max_features':trial.suggest_int('max_features',2,4,1)
    }
    return params

In [27]:
factory = PreprocessorFactory(conts, cats, catOrders)
selected_features = ['sleep_duration', 'stress_level', 'physical_activity_level', 'bmi']
dynamic_preprocessor = factory.build(selected_features)

rf_tuner = Learner(train, test, target, RandomForestClassifier, selected_features, 'RF_Tuner_4features_2', balanced_accuracy_score, preprocessor=dynamic_preprocessor)

best_params = rf_tuner.fast_gpu_tune(study_name='RF_GPU_Tuning_4f', n_trials=30, get_params_func=rf_param_space, db_path='/content/drive/MyDrive/KaggleData/s6e7/rf_gpu_tuning_study.db')

[I 2026-07-22 02:13:52,705] Using an existing study with name 'RF_GPU_Tuning_4f' instead of creating a new one.


  0%|          | 0/30 [00:00<?, ?it/s]

🏃 View run Trial_50 at: https://dbc-36754e4d-ad0b.cloud.databricks.com/ml/experiments/1584872737403196/runs/a449cdbbe7124e93b0e4b7d77b35a658
🧪 View experiment at: https://dbc-36754e4d-ad0b.cloud.databricks.com/ml/experiments/1584872737403196
[I 2026-07-22 02:15:01,384] Trial 50 finished with value: 0.86552921976531 and parameters: {'n_estimators': 200, 'max_depth': 25, 'min_samples_leaf': 18, 'max_features': 4}. Best is trial 50 with value: 0.86552921976531.
🏃 View run Trial_51 at: https://dbc-36754e4d-ad0b.cloud.databricks.com/ml/experiments/1584872737403196/runs/7e4c71cb0537461a95d7e4cc64ffaaec
🧪 View experiment at: https://dbc-36754e4d-ad0b.cloud.databricks.com/ml/experiments/1584872737403196
[I 2026-07-22 02:16:18,367] Trial 51 finished with value: 0.871615398168742 and parameters: {'n_estimators': 200, 'max_depth': 25, 'min_samples_leaf': 2, 'max_features': 4}. Best is trial 51 with value: 0.871615398168742.
🏃 View run Trial_52 at: https://dbc-36754e4d-ad0b.cloud.databricks.com/ml

In [30]:
best_params_1 = {'n_estimators': 75,
 'max_depth': 20,
 'min_samples_leaf': 10,
 'max_features': 4}

In [31]:
best_params_2 = {'n_estimators': 150,
 'max_depth': 35,
 'min_samples_leaf': 2,
 'max_features': 4}

In [37]:
%%time

factory = PreprocessorFactory(conts, cats, catOrders)
selected_features = ['sleep_duration', 'stress_level', 'physical_activity_level', 'bmi']
dynamic_preprocessor = factory.build(selected_features)

rf_model = Learner(train, test, target, RandomForestClassifier, selected_features, 'RF_Subset_top_features_4_tuned_bp2', balanced_accuracy_score, preprocessor=dynamic_preprocessor)
rf_model.fit(trainingMode='standard', params=best_params_2)

Training fold 1...
Fold 1 ==> OOF score: 0.873090170799513
Training fold 2...
Fold 2 ==> OOF score: 0.8752171059091727
Training fold 3...
Fold 3 ==> OOF score: 0.8743572945921748
Training fold 4...
Fold 4 ==> OOF score: 0.8724156419685661
Training fold 5...
Fold 5 ==> OOF score: 0.8733937509047968
CV Mean Accuracy: 0.8737 | Std: 0.0010
---------------------------------------------------------------
🏃 View run RF_Subset_top_features_4_tuned_bp2_standard at: https://dbc-36754e4d-ad0b.cloud.databricks.com/ml/experiments/1584872737403196/runs/3606fb3df9084b058411e5a92cc329a3
🧪 View experiment at: https://dbc-36754e4d-ad0b.cloud.databricks.com/ml/experiments/1584872737403196
CPU times: user 1min 1s, sys: 24.3 s, total: 1min 25s
Wall time: 1min 21s


np.float64(0.8736947928348447)

In [38]:
submit_preds(rf_model, target, 'RF_Subset_top_features_4_tuned_bp2.csv')

In [29]:
import plotly.graph_objects as go

import plotly.io as pio

# Force Plotly to render natively inside VS Code
pio.renderers.default = "vscode"
experiment_path = f"/Users/{user_email}/Kaggle_s6e7_Comp"
experiment = mlflow.get_experiment_by_name(experiment_path)

if experiment is None:
    raise ValueError(f"Could not find experiment at {experiment_path}")

# 2. Search for all runs inside this experiment
runs_df = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

# 3. Filter down to child trial runs only
child_runs = runs_df[runs_df['tags.mlflow.parentRunId'].notna()].copy()

# Extract numeric trial number from run name (e.g. "Trial_5" -> 5)
child_runs['trial_number'] = child_runs['tags.mlflow.runName'].str.replace('Trial_', '').astype(int)

# Sort chronologically by trial number
child_runs = child_runs.sort_values('trial_number')

# 4. Create the Overfitting Line Plot
fig = go.Figure()

# Add Training Score Line
fig.add_trace(go.Scatter(
    x=child_runs['trial_number'], 
    y=child_runs['metrics.cv_mean_train_acc'],
    mode='lines+markers',
    name='CV Mean Train Acc',
    line=dict(color='orange', width=2)
))

# Add Validation Score Line
fig.add_trace(go.Scatter(
    x=child_runs['trial_number'], 
    y=child_runs['metrics.cv_mean_val_acc'],
    mode='lines+markers',
    name='CV Mean Val Acc',
    line=dict(color='teal', width=2)
))

fig.update_layout(
    title='Train vs. Validation Score per Optuna Trial (Overfitting Check)',
    xaxis_title='Trial Number',
    yaxis_title='Balanced Accuracy',
    hovermode='x unified',
    template='plotly_white',
    legend=dict(x=0.01, y=0.01)
)

fig.show()

In [118]:
best_params = {'n_estimators': 500, 'max_depth': 19, 'min_samples_leaf': 10, 'max_features': 15, 'n_jobs': -1, 'random_state': 42}
rf = learner(train,test,target,RandomForestClassifier,feature_cols,'TunedRF-Allcols-5Fold', balanced_accuracy_score)
rf.fit('None', params=best_params)

---------------------------------------------------------------
Fold, 1, ==> TunedRF-Allcols-5Fold OOF score is ==> 0.9613524073316425
---------------------------------------------------------------
Fold, 2, ==> TunedRF-Allcols-5Fold OOF score is ==> 0.9634411557118496
---------------------------------------------------------------
Fold, 3, ==> TunedRF-Allcols-5Fold OOF score is ==> 0.9627244001316848
---------------------------------------------------------------
Fold, 4, ==> TunedRF-Allcols-5Fold OOF score is ==> 0.9627292114397142
---------------------------------------------------------------
Fold, 5, ==> TunedRF-Allcols-5Fold OOF score is ==> 0.9622523053271634
---------------------------------------------------------------
Average OOF Accuracy of model is: 0.962499895988411
--------------------------------------------------------------
                       avgOOFAcc
BasicRF-Allcols         0.960187
TunedRF-Allcols-5Fold   0.962500


In [135]:
submit_preds(rf, sample_submission, target, 'TunedRF5Fold.csv')